# COMP90016 JupyterLite Environment Check (Unified)

This notebook uses one unified checker for Python packages (including native-extension packages), then checks CLI tool callability separately.

In [ ]:
import sys

CHECKS = [
    {'module': 'numpy', 'dist': 'numpy', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'numpy'},
    {'module': 'pandas', 'dist': 'pandas', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'pandas'},
    {'module': 'matplotlib', 'dist': 'matplotlib', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'matplotlib'},
    {'module': 'seaborn', 'dist': 'seaborn', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'seaborn'},
    {'module': 'Bio', 'dist': 'biopython', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'bio'},
    {'module': 'plotly', 'dist': 'plotly', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'plotly'},
    {'module': 'bqplot', 'dist': 'bqplot', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'bqplot'},
    {'module': 'itables', 'dist': 'itables', 'expected_in_pyodide': True, 'install_via_piplite': True, 'smoke': 'itables'},
    {'module': 'fastcluster', 'dist': 'fastcluster', 'expected_in_pyodide': False, 'install_via_piplite': False, 'smoke': 'default'},
    {'module': 'pysam', 'dist': 'pysam', 'expected_in_pyodide': False, 'install_via_piplite': False, 'smoke': 'default'},
]

if 'pyodide' in sys.modules:
    import piplite
    to_install = [item['dist'] for item in CHECKS if item['install_via_piplite']]
    await piplite.install(to_install, keep_going=True)
    print('Pyodide detected. Requested package install for browser-target set:')
    print(to_install)
else:
    print('Pyodide not detected; skipped piplite install step.')

In [ ]:
import importlib
import importlib.metadata as md
import sys

def unload(module_name):
    for key in list(sys.modules):
        if key == module_name or key.startswith(module_name + '.'):
            sys.modules.pop(key, None)
    importlib.invalidate_caches()

def smoke_numpy(_module):
    import numpy as np
    return int(np.array([1, 2, 3]).sum()) == 6

def smoke_pandas(_module):
    import pandas as pd
    return int(pd.DataFrame({'x': [1, 2, 3]})['x'].sum()) == 6

def smoke_matplotlib(_module):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    ax.plot([0, 1], [0, 1])
    plt.close(fig)
    return True

def smoke_seaborn(_module):
    import seaborn as sns
    sns.set_theme()
    return True

def smoke_bio(_module):
    from Bio.Seq import Seq
    return str(Seq('ATGC').reverse_complement()) == 'GCAT'

def smoke_plotly(_module):
    import plotly.graph_objects as go
    go.Figure()
    return True

def smoke_bqplot(_module):
    from bqplot import Figure, LinearScale
    Figure(scales={'x': LinearScale(), 'y': LinearScale()})
    return True

def smoke_itables(_module):
    from itables import show
    _ = show
    return True

def smoke_default(_module):
    return True

SMOKE_FUNCS = {
    'numpy': smoke_numpy,
    'pandas': smoke_pandas,
    'matplotlib': smoke_matplotlib,
    'seaborn': smoke_seaborn,
    'bio': smoke_bio,
    'plotly': smoke_plotly,
    'bqplot': smoke_bqplot,
    'itables': smoke_itables,
    'default': smoke_default,
}

print('Unified cold functionality check (Python packages)')
for item in CHECKS:
    module_name = item['module']
    dist_name = item['dist']
    expected_available = item['expected_in_pyodide']
    smoke = SMOKE_FUNCS[item['smoke']]

    expected = 'AVAILABLE' if expected_available else 'UNAVAILABLE'

    try:
        dist_version = md.version(dist_name)
    except md.PackageNotFoundError:
        dist_version = '-'

    observed = 'UNAVAILABLE'
    detail = ''
    try:
        unload(module_name)
        module = importlib.import_module(module_name)
        smoke(module)
        observed = 'AVAILABLE'
        detail = getattr(module, '__version__', dist_version)
    except Exception as exc:
        detail = f'{type(exc).__name__}: {exc}'

    result = 'OK' if observed == expected else 'MISMATCH'
    print(f"{dist_name:12} expected={expected:11} observed={observed:11} result={result:8} {detail}")

In [ ]:
import shutil
import subprocess

cli_tools = [
    'blast',
    'bwa',
    'fastqc',
    'fastqe',
    'iqtree',
    'mlst',
    'prokka',
    'samtools',
    'snp-dists',
]

print('\nCLI tools callability check (expected: NOT CALLABLE in JupyterLite)')
for tool in cli_tools:
    tool_path = shutil.which(tool)
    if tool_path is None:
        print(f'{tool:12} NOT CALLABLE not found on PATH')
        continue

    try:
        proc = subprocess.run([tool, '--version'], capture_output=True, text=True, timeout=5)
        if proc.returncode == 0:
            head = (proc.stdout or proc.stderr or '').splitlines()
            snippet = head[0] if head else 'version command succeeded'
            print(f'{tool:12} CALLABLE     {snippet}')
        else:
            print(f'{tool:12} NOT CALLABLE returncode={proc.returncode}')
    except Exception as exc:
        print(f'{tool:12} NOT CALLABLE {type(exc).__name__}: {exc}')